# Multimodal Fusion: Gated (Highway) Fusion

This notebook implements **Gated (Highway) Fusion** for in-hospital mortality prediction.
- **Temporal Features**: $t \in \mathbb{R}^{64}$
- **Static Tabular Features**: $s \in \mathbb{R}^{64}$
- **Adaptive Gating Mechanism**: A learnable gate network computes dynamic modality weights:
  $$g = \sigma(W [t \,\|\, s] + b) \in [0, 1]^{64}$$
  $$X_{\text{fused}} = g \odot t + (1 - g) \odot s \in \mathbb{R}^{64}$$
  allowing the model to adaptively emphasize temporal trajectory vs static baseline per patient dimension.


## Step 1: Imports, Configuration & Device Setup
Import dependencies, configure random seeds, and select target compute device (CUDA / MPS / CPU).


In [2]:
from pathlib import Path
import copy
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, accuracy_score,
    balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

SEED = 42
BATCH_SIZE = 512
MAX_EPOCHS = 50
PATIENCE = 7
LR = 1e-3

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

_start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_start, *_start.parents] if (p / "checkpoints").exists() and (p / "processed").exists()), _start)
FUSION_DIR = PROJECT_ROOT / "checkpoints" / "fusion"
OUT_DIR = FUSION_DIR / "gated_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Output:", OUT_DIR.resolve())




Device: mps
Output: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gated_baseline


## Step 2: Load Multimodal Embeddings & Cohort Validation
Load temporal and tabular embeddings from `fused_embeddings_{split}.npz` and ensure patient disjointness across splits.


In [3]:
def load_split(split):
    path = FUSION_DIR / f"fused_embeddings_{split}.npz"

    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path) as data:
        temporal = data["temporal_embeddings"].astype(np.float32)
        tabular = data["tabular_embeddings"].astype(np.float32)
        ids = data["patient_ids"].reshape(-1).astype(np.int64)
        labels = data["labels"].reshape(-1).astype(np.float32)

    assert temporal.shape == tabular.shape
    assert temporal.ndim == 2 and temporal.shape[1] == 64
    assert len(temporal) == len(ids) == len(labels)
    assert len(np.unique(ids)) == len(ids)
    assert np.isfinite(temporal).all()
    assert np.isfinite(tabular).all()
    assert np.isfinite(labels).all()
    assert set(np.unique(labels)).issubset({0.0, 1.0})

    return temporal, tabular, labels, ids


t_train, s_train, y_train, ids_train = load_split("train")
t_val, s_val, y_val, ids_val = load_split("val")
t_test, s_test, y_test, ids_test = load_split("test")

assert not (set(ids_train) & set(ids_val))
assert not (set(ids_train) & set(ids_test))
assert not (set(ids_val) & set(ids_test))

print("Train:", t_train.shape)
print("Validation:", t_val.shape)
print("Test:", t_test.shape)



Train: (153831, 64)
Validation: (24186, 64)
Test: (22809, 64)


## Step 3: Modality-Specific Feature Standardization
Fit separate standardizers for temporal and tabular modalities on the training split only.


In [4]:
# Standardize both modalities using training statistics only.
t_mean, t_std = t_train.mean(axis=0), t_train.std(axis=0)
s_mean, s_std = s_train.mean(axis=0), s_train.std(axis=0)

t_std[t_std < 1e-8] = 1.0
s_std[s_std < 1e-8] = 1.0

t_train = ((t_train - t_mean) / t_std).astype(np.float32)
t_val = ((t_val - t_mean) / t_std).astype(np.float32)
t_test = ((t_test - t_mean) / t_std).astype(np.float32)

s_train = ((s_train - s_mean) / s_std).astype(np.float32)
s_val = ((s_val - s_mean) / s_std).astype(np.float32)
s_test = ((s_test - s_mean) / s_std).astype(np.float32)

assert np.isfinite(t_train).all() and np.isfinite(s_train).all()
assert np.isfinite(t_val).all() and np.isfinite(s_val).all()
assert np.isfinite(t_test).all() and np.isfinite(s_test).all()

np.savez(
    OUT_DIR / "input_scaler.npz",
    temporal_mean=t_mean,
    temporal_std=t_std,
    tabular_mean=s_mean,
    tabular_std=s_std
)




## Step 4: PyTorch DataLoaders & Class Balancing
Construct PyTorch DataLoaders for paired modalities and compute positive class loss weight.


In [5]:
def make_loader(t, s, y, shuffle):
    dataset = TensorDataset(
        torch.from_numpy(t),
        torch.from_numpy(s),
        torch.from_numpy(y.astype(np.float32))
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0
    )


train_loader = make_loader(t_train, s_train, y_train, True)
val_loader = make_loader(t_val, s_val, y_val, False)
test_loader = make_loader(t_test, s_test, y_test, False)




## Step 5: Gated Fusion Classifier Architecture
Define `GatedFusionClassifier` computing soft gating weights and fused representations, followed by linear classification.


In [6]:
class GatedFusionClassifier(nn.Module):
    def __init__(self, embedding_dim=64, dropout=0.25):
        super().__init__()
        self.gate = nn.Linear(embedding_dim * 2, embedding_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, temporal, tabular):
        gate = torch.sigmoid(
            self.gate(torch.cat([temporal, tabular], dim=1))
        )
        fused = gate * temporal + (1.0 - gate) * tabular
        logits = self.classifier(fused).squeeze(-1)
        return logits, gate


model = GatedFusionClassifier().to(device)

n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        n_neg / max(n_pos, 1),
        dtype=torch.float32,
        device=device
    )
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

print(f"Training positives: {n_pos:,}")
print(f"Training negatives: {n_neg:,}")


def predict_probabilities(loader):
    model.eval()
    labels, probabilities = [], []

    with torch.no_grad():
        for temporal, tabular, y in loader:
            temporal = temporal.to(device)
            tabular = tabular.to(device)

            logits, _ = model(temporal, tabular)
            probabilities.append(
                torch.sigmoid(logits).cpu().numpy()
            )
            labels.append(y.numpy())

    return np.concatenate(labels), np.concatenate(probabilities)




Training positives: 13,634
Training negatives: 140,197


## Step 6: Training Loop with Validation AUPRC Early Stopping
Optimize model weights with early stopping based on validation AUPRC. Save best checkpoint to `gated_classifier.pt`.


In [7]:
best_val_ap = -np.inf
best_state = None
wait = 0
history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for temporal, tabular, labels in train_loader:
        temporal = temporal.to(device)
        tabular = tabular.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(temporal, tabular)
        loss = criterion(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        total_samples += len(labels)

    train_loss = total_loss / total_samples
    val_labels, val_probs = predict_probabilities(val_loader)

    val_ap = average_precision_score(val_labels, val_probs)
    val_auc = roc_auc_score(val_labels, val_probs)

    history.append({
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_auprc": float(val_ap),
        "val_auroc": float(val_auc)
    })

    print(
        f"Epoch {epoch + 1:02d}/{MAX_EPOCHS} | "
        f"Train loss={train_loss:.4f} | "
        f"Val AUPRC={val_ap:.4f} | "
        f"Val AUROC={val_auc:.4f}"
    )

    if np.isfinite(val_ap) and val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("No valid checkpoint selected.")

model.load_state_dict(best_state)

torch.save(
    model.state_dict(),
    OUT_DIR / "gated_classifier.pt"
)

with open(OUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"Best validation AUPRC: {best_val_ap:.4f}")



Epoch 01/50 | Train loss=0.7473 | Val AUPRC=0.5205 | Val AUROC=0.8748
Epoch 02/50 | Train loss=0.6994 | Val AUPRC=0.5209 | Val AUROC=0.8763
Epoch 03/50 | Train loss=0.6926 | Val AUPRC=0.5247 | Val AUROC=0.8785
Epoch 04/50 | Train loss=0.6875 | Val AUPRC=0.5237 | Val AUROC=0.8776
Epoch 05/50 | Train loss=0.6829 | Val AUPRC=0.5225 | Val AUROC=0.8780
Epoch 06/50 | Train loss=0.6801 | Val AUPRC=0.5258 | Val AUROC=0.8790
Epoch 07/50 | Train loss=0.6771 | Val AUPRC=0.5267 | Val AUROC=0.8789
Epoch 08/50 | Train loss=0.6758 | Val AUPRC=0.5259 | Val AUROC=0.8787
Epoch 09/50 | Train loss=0.6728 | Val AUPRC=0.5255 | Val AUROC=0.8798
Epoch 10/50 | Train loss=0.6716 | Val AUPRC=0.5263 | Val AUROC=0.8792
Epoch 11/50 | Train loss=0.6689 | Val AUPRC=0.5258 | Val AUROC=0.8797
Epoch 12/50 | Train loss=0.6657 | Val AUPRC=0.5227 | Val AUROC=0.8781
Epoch 13/50 | Train loss=0.6650 | Val AUPRC=0.5256 | Val AUROC=0.8799
Epoch 14/50 | Train loss=0.6632 | Val AUPRC=0.5227 | Val AUROC=0.8779
Early stopping.
Best

## Step 7: Optimal Decision Threshold Selection (Validation F1)
Sweep classification thresholds over the validation PR curve to select the threshold maximizing F1 score.


In [8]:
# Select the threshold using validation F1, not test results.
val_labels, val_probs = predict_probabilities(val_loader)
p_curve, r_curve, thresholds = precision_recall_curve(
    val_labels, val_probs
)

f1_curve = (
    2 * p_curve[:-1] * r_curve[:-1]
    / (p_curve[:-1] + r_curve[:-1] + 1e-12)
)

best_idx = int(np.argmax(f1_curve))
best_threshold = float(thresholds[best_idx])

print(f"Selected threshold: {best_threshold:.4f}")
print(f"Validation F1: {f1_curve[best_idx]:.4f}")



Selected threshold: 0.7826
Validation F1: 0.5018


## Step 8: Held-Out Test Evaluation & Metrics Export
Evaluate the test set once with the selected threshold, format confusion matrix, and export `test_metrics.json`.


In [9]:
# Evaluate the test set once using the selected threshold.
test_labels, test_probs = predict_probabilities(test_loader)
test_pred = (test_probs >= best_threshold).astype(int)

results = {
    "fusion": "gated",
    "classifier": "MLP",
    "threshold": best_threshold,
    "test_auroc": float(roc_auc_score(test_labels, test_probs)),
    "test_auprc": float(
        average_precision_score(test_labels, test_probs)
    ),
    "test_accuracy": float(
        accuracy_score(test_labels, test_pred)
    ),
    "test_balanced_accuracy": float(
        balanced_accuracy_score(test_labels, test_pred)
    ),
    "test_precision": float(
        precision_score(test_labels, test_pred, zero_division=0)
    ),
    "test_recall": float(
        recall_score(test_labels, test_pred, zero_division=0)
    ),
    "test_f1": float(
        f1_score(test_labels, test_pred, zero_division=0)
    ),
    "confusion_matrix": confusion_matrix(
        test_labels, test_pred, labels=[0, 1]
    ).tolist()
}

print("\nGATED FUSION RESULTS")
for key, value in results.items():
    print(f"{key}: {value}")

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)




GATED FUSION RESULTS
fusion: gated
classifier: MLP
threshold: 0.7826394438743591
test_auroc: 0.874072643336608
test_auprc: 0.527044648499254
test_accuracy: 0.9054759086325573
test_balanced_accuracy: 0.7279552635810465
test_precision: 0.4829622898682417
test_recall: 0.5108121095627103
test_f1: 0.4964969640354974
confusion_matrix: [[19590, 1138], [1018, 1063]]


## Step 9: Gate Distribution & Modality Contribution Analysis
Extract learned gate activations across all test patients to quantify the relative contribution of temporal vs static modalities.


In [10]:
# Summarize which modality the learned gate favored on test patients.
model.eval()
gate_values = []

with torch.no_grad():
    for temporal, tabular, _ in test_loader:
        _, gates = model(temporal.to(device), tabular.to(device))
        gate_values.append(gates.cpu().numpy())

gate_values = np.concatenate(gate_values, axis=0)

print(f"Mean temporal gate weight: {gate_values.mean():.4f}")
print(f"Mean tabular gate weight: {1.0 - gate_values.mean():.4f}")
print("Saved results to:", OUT_DIR.resolve())


Mean temporal gate weight: 0.5009
Mean tabular gate weight: 0.4991
Saved results to: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/gated_baseline
